In [1]:
import biotrade
import geopandas as gpd
from shapely.geometry import Polygon


Create /home/jan/repos/forobs/biotrade_data?Please confirm [y/n]:y


In [5]:
from biotrade.faostat import faostat
faostat.pump.update(["crop_production"])


If the database Engine(sqlite:////home/jan/repos/forobs/biotrade_data/faostat/faostat.db) exists already, this command will erase the following tables and replace them with new data:
 - crop_production
Please confirm [y/n]:y


2024-03-22 16:40:49,786 - biotrade.faostat - INFO - Downloading data from:
 http://fenixservices.fao.org/faostat/static/bulkdownloads/Production_Crops_Livestock_E_All_Data_(Normalized).zip


HTTP response code: 200


/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/biotrade/faostat/pump.py:323: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_chunk in pandas.read_csv(


   reporter_code     reporter  product_code           product  element_code  \
0              2  Afghanistan           221  almonds_in_shell          5312   

          element  period  year unit  value flag  
0  area_harvested    1961  1961   ha    0.0    A  


2024-03-22 16:41:25,671 - biotrade.comtrade - INFO - Wrote 4127584 rows to the database table crop_production


In [7]:
from biotrade.common.aggregate import nlargest
wheat = faostat.db.select("crop_production", product="wheat")
wheat_largest = nlargest(
    # Remove aggregates (continents) and keep only recent years
    wheat.query("reporter_code < 1000 and year > year.max() - 10"),
    value_vars="value",
    agg_groups=["reporter", "product"],
    slice_groups=["element", "unit"],
    n=5
    )

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/biotrade/common/aggregate.py:290: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df_agg = df.groupby(agg_groups + time_vars)[value_vars].agg(sum)


In [17]:
crops = faostat.db.select("crop_production")


In [32]:
crops_2020 = crops.loc[crops.year == 2020].reset_index(drop=True)

In [37]:
crops_2020_area = crops_2020.loc[crops_2020.element == 'area_harvested'].reset_index(drop=True)

In [43]:
crops_2020_area['product'].unique()

array(['almonds_in_shell',
       'anise_badian_coriander_cumin_caraway_fennel_and_juniper_berries_raw',
       'apples', 'apricots', 'barley', 'cantaloupes_and_other_melons',
       'figs', 'grapes', 'linseed', 'maize_corn', 'millet',
       'mustard_seed', 'olives',
       'onions_and_shallots_dry_excluding_dehydrated', 'oranges',
       'other_berries_and_fruits_of_the_genus_vaccinium_n_e_c',
       'other_citrus_fruit_n_e_c', 'other_fruits_n_e_c',
       'other_nuts_excluding_wild_edible_nuts_and_groundnuts_in_shell_n_e_c',
       'other_pulses_n_e_c',
       'other_stimulant_spice_and_aromatic_crops_n_e_c',
       'other_stone_fruits', 'other_vegetables_fresh_n_e_c',
       'peaches_and_nectarines', 'pears', 'pistachios_in_shell',
       'plums_and_sloes', 'potatoes', 'rice', 'seed_cotton_unginned',
       'sesame_seed', 'sugar_beet', 'sugar_cane', 'sunflower_seed',
       'walnuts_in_shell', 'watermelons', 'wheat', 'cereals_primary',
       'citrus_fruit_total', 'fibre_crops_fibr

In [20]:

gdf = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

/tmp/ipykernel_687910/4051158862.py:3: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  gdf = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [24]:
min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])


In [26]:
tropical_countries = gdf.loc[gdf.intersects(tropics)]